# Pemodelan & Simulasi Penyebaran Hoaks di Media Sosial (Model S-I-R)

Mata kuliah: Pemodelan Sistem Dinamik  
Nama / NIM: ........................

**Persamaan model:**

$$\frac{dS}{dt}=-(1-u)\beta S I,\quad \frac{dI}{dt}=(1-u)\beta S I-\gamma I,\quad \frac{dR}{dt}=\gamma I$$

- S: pengguna belum terpapar hoaks
- I: penyebar aktif
- R: pengguna yang sudah berhenti menyebarkan
- u: strategi kontrol (literasi digital & klarifikasi), 0 ≤ u ≤ 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint

## 1. Parameter dan kondisi awal

In [ ]:
N     = 10000   # total pengguna
beta  = 6e-5    # laju kontak-penyebaran (1/(pengguna*hari))
gamma = 0.2     # laju berhenti menyebar (1/hari) -> rata-rata 5 hari

S0, I0, R0_ = 9990, 10, 0
x0 = [S0, I0, R0_]

print('R0 =', beta*N/gamma)

## 2. Fungsi model

In [ ]:
def hoaks(x, t, u):
    S, I, R = x
    dS = -(1 - u) * beta * S * I
    dI =  (1 - u) * beta * S * I - gamma * I
    dR =  gamma * I
    return [dS, dI, dR]

## 3. Simulasi tanpa kontrol (u = 0)

In [ ]:
t = np.linspace(0, 120, 1201)
x = odeint(hoaks, x0, t, args=(0.0,))
S, I, R = x[:, 0], x[:, 1], x[:, 2]

plt.figure(figsize=(9, 5))
plt.plot(t, S, 'b-', label='S (belum terpapar)')
plt.plot(t, I, 'r-', label='I (penyebar aktif)')
plt.plot(t, R, 'g-', label='R (berhenti menyebar)')
plt.xlabel('Waktu (hari)'); plt.ylabel('Jumlah pengguna')
plt.title('Penyebaran Hoaks Tanpa Kontrol (u = 0)')
plt.grid(alpha=0.3); plt.legend()
plt.show()

## 4. Variasi strategi kontrol u

In [ ]:
plt.figure(figsize=(9, 5))
for uu, st in [(0.0, 'r-'), (0.3, 'b--'), (0.5, 'g-.'), (0.7, 'm:')]:
    xx = odeint(hoaks, x0, t, args=(uu,))
    plt.plot(t, xx[:, 1], st, linewidth=2, label=f'u = {uu}')
plt.xlabel('Waktu (hari)'); plt.ylabel('Penyebar aktif I(t)')
plt.title('Pengaruh Literasi Digital (u) terhadap Kurva Penyebar')
plt.grid(alpha=0.3); plt.legend()
plt.show()

## 5. Perhitungan manual (metode Euler, Δt = 1 hari) dan perbandingan dengan odeint

In [ ]:
dt = 1.0
S_e, I_e, R_e = [S0], [I0], [R0_]
for k in range(5):
    dS, dI, dR = hoaks([S_e[-1], I_e[-1], R_e[-1]], 0, 0.0)
    S_e.append(S_e[-1] + dt * dS)
    I_e.append(I_e[-1] + dt * dI)
    R_e.append(R_e[-1] + dt * dR)

xo = odeint(hoaks, x0, np.arange(0, 6), args=(0.0,))
print('hari |  S_euler  I_euler  R_euler | I_odeint')
for k in range(6):
    print(f'{k:4d} | {S_e[k]:8.2f} {I_e[k]:8.2f} {R_e[k]:8.2f} | {xo[k,1]:8.2f}')

plt.figure(figsize=(9, 5))
plt.plot(np.arange(0, 6), I_e, 'ro--', label='I Euler (dt = 1 hari)')
plt.plot(np.arange(0, 6), xo[:, 1], 'b-', label='I odeint')
plt.xlabel('Waktu (hari)'); plt.ylabel('Penyebar aktif I(t)')
plt.title('Perbandingan Perhitungan Manual (Euler) dan odeint')
plt.grid(alpha=0.3); plt.legend()
plt.show()

## 6. Ringkasan hasil

In [ ]:
for uu in [0.0, 0.3, 0.5, 0.7]:
    xx = odeint(hoaks, x0, t, args=(uu,))
    ip = xx[:, 1].argmax()
    print(f'u={uu}: puncak I={xx[ip,1]:.0f} pada hari {t[ip]:.1f}; '
          f'R akhir={xx[-1,2]:.0f}; S akhir={xx[-1,0]:.0f}')